# 01 · Ingesta desde raw (H1.1 · H1.2 · H1.3)
Este notebook funciona solo: no depende de otros archivos del repo.

1. **Lectura (H1.1):** con `pd.read_csv(URL)`, sin descargar archivos, los **1.143 reportes diarios**
   (22-ene-2020 → 9-mar-2023), las **5 series de tiempo** y la **tabla UID**.
2. **Unificación (H1.2):** los 5 esquemas en una sola tabla con nombres en snake_case.
3. **Llave de territorio (H1.3):** nombres de país normalizados y cruce con la tabla UID.
4. **Capa limpia (H1.4):** la tabla se guarda en Parquet y se compara con CSV.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

RAIZ = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/"
BASE = RAIZ + "csse_covid_19_daily_reports/"
BASE_SERIES = RAIZ + "csse_covid_19_time_series/"

# Reportes diarios: del 22-ene-2020 al 9-mar-2023 = 1.143 días.
fechas = pd.date_range("2020-01-22", "2023-03-09")

## 1 · Lectura (H1.1)
**Por defecto, con hilos:** 10 archivos a la vez, ~20 s.
La celda siguiente (comentada) es la versión sin hilos, como la PPT S14: ~6 min. *Pendiente decidir en grupo cuál dejar.*

In [ ]:
def leer_reporte(f):
    """Lee el reporte de una fecha tal como viene (columnas originales)."""
    nombre = f.strftime("%m-%d-%Y") + ".csv"   # el archivo se llama mes-día-año
    return pd.read_csv(BASE + nombre)


with ThreadPoolExecutor(max_workers=10) as ex:
    reportes = dict(zip(fechas, ex.map(leer_reporte, fechas)))   # map respeta el orden de las fechas

In [ ]:
# Alternativa sin hilos (como la PPT S14), ~6 min:
# reportes = {}
# for f in fechas:
#     reportes[f] = leer_reporte(f)

In [ ]:
series = {
    "confirmados_global": pd.read_csv(BASE_SERIES + "time_series_covid19_confirmed_global.csv"),
    "fallecidos_global": pd.read_csv(BASE_SERIES + "time_series_covid19_deaths_global.csv"),
    "recuperados_global": pd.read_csv(BASE_SERIES + "time_series_covid19_recovered_global.csv"),
    "confirmados_us": pd.read_csv(BASE_SERIES + "time_series_covid19_confirmed_US.csv"),
    "fallecidos_us": pd.read_csv(BASE_SERIES + "time_series_covid19_deaths_US.csv"),
}
uid = pd.read_csv(RAIZ + "UID_ISO_FIPS_LookUp_Table.csv")   # llave de territorio, ISO3 y población

### Control: se leyeron las 1.143 fechas

In [ ]:
print("Fechas esperadas:", len(fechas), "| leídas:", len(reportes))
print("Filas en total:", sum(len(t) for t in reportes.values()))
for nombre, t in series.items():
    print(f"Serie {nombre}: {len(t)} filas")
print("Tabla UID:", len(uid), "filas")

### Cambios de esquema
Agrupamos los reportes por su lista de columnas: cada grupo es un esquema.

In [ ]:
columnas = pd.Series({f: ", ".join(t.columns) for f, t in reportes.items()})
columnas.groupby(columnas, sort=False).apply(lambda s: f"{s.index.min().date()} → {s.index.max().date()} ({len(s)} reportes)")

## 2 · Unificación de los 5 esquemas (H1.2)
El diccionario `RENOMBRAR` (PPT S14, lámina 26) traduce cada nombre antiguo a uno solo en snake_case. Además se agregan:
- `fecha_reporte`: la fecha del nombre del archivo (no `last_update`, que es cuándo JHU tocó la fila);
- `esquema`: 1 a 5, para saber de qué etapa viene cada fila;
- `archivo_origen` y `fila_origen` (**linaje**, H2.6): de qué CSV y de qué fila salió cada registro.
  `fila_origen` es la posición en `pd.read_csv` (0 = primera fila de datos), así se comprueba con
  `pd.read_csv(BASE + archivo_origen).iloc[fila_origen]`.

Las columnas que no existen en una etapa quedan vacías: no se inventan.

In [ ]:
RENOMBRAR = {
    "Province/State": "province_state", "Province_State": "province_state",
    "Country/Region": "country_region", "Country_Region": "country_region",
    "Last Update": "last_update", "Last_Update": "last_update",
    "Latitude": "lat", "Lat": "lat", "Longitude": "long", "Long_": "long",
    "Incidence_Rate": "incident_rate", "Incident_Rate": "incident_rate",
    "Case-Fatality_Ratio": "case_fatality_ratio",
    "Case_Fatality_Ratio": "case_fatality_ratio",
}

# Fecha en que empieza cada esquema, del 2 al 5 (el 1 parte el 22-ene-2020).
CORTES = ["2020-03-01", "2020-03-22", "2020-05-29", "2020-11-09"]

tablas = []
for f, t in reportes.items():
    t = t.rename(columns=RENOMBRAR)
    t.columns = t.columns.str.lower()
    t["fecha_reporte"] = f                                         # fecha del archivo, no last_update
    t["esquema"] = 1 + sum(f >= pd.Timestamp(c) for c in CORTES)   # 1 a 5
    t["archivo_origen"] = f.strftime("%m-%d-%Y") + ".csv"         # linaje: archivo...
    t["fila_origen"] = t.index                                     # ...y fila dentro del archivo
    tablas.append(t)

df = pd.concat(tablas, ignore_index=True)
df["last_update"] = pd.to_datetime(df["last_update"], format="mixed").dt.normalize()   # AAAA-MM-DD

print("Filas:", len(df), "| columnas:", df.shape[1])
df.dtypes

### Control: ninguna columna gemela

In [ ]:
gemelas = [c for c in df.columns if c in ("long_", "incidence_rate", "case-fatality_ratio", "latitude", "longitude")]
print("Columnas gemelas:", gemelas or "ninguna")
print("Filas unificadas = filas leídas:", len(df) == sum(len(t) for t in reportes.values()))

### Filas por esquema

In [ ]:
df.groupby("esquema").agg(desde=("fecha_reporte", "min"), hasta=("fecha_reporte", "max"),
                          reportes=("fecha_reporte", "nunique"), filas=("fecha_reporte", "size"))

## 3 · Llave de territorio (H1.3)
JHU escribió algunos países de varias formas (`Mainland China` → `China`, `South Korea` → `Korea, South`), y algunos
territorios partieron como país y después pasaron a ser provincia de otro (`Hong Kong` → provincia de `China`).
Los diccionarios `PAIS` y `TERRITORIO` (los mismos de `00_exploracion`) unifican esos nombres.

In [ ]:
PAIS = {
    " Azerbaijan": "Azerbaijan", "Bahamas, The": "Bahamas", "The Bahamas": "Bahamas",
    "Cape Verde": "Cabo Verde", "Czech Republic": "Czechia", "East Timor": "Timor-Leste",
    "Gambia, The": "Gambia", "The Gambia": "Gambia", "Iran (Islamic Republic of)": "Iran",
    "Ivory Coast": "Cote d'Ivoire", "Mainland China": "China", "Republic of Ireland": "Ireland",
    "Republic of Korea": "Korea, South", "South Korea": "Korea, South",
    "Republic of Moldova": "Moldova", "Republic of the Congo": "Congo (Brazzaville)",
    "Russian Federation": "Russia", "Taiwan": "Taiwan*", "Taipei and environs": "Taiwan*",
    "Vatican City": "Holy See", "Viet Nam": "Vietnam",
    "Palestine": "West Bank and Gaza", "occupied Palestinian territory": "West Bank and Gaza",
    "North Ireland": "United Kingdom", "Others": "Diamond Princess", "Cruise Ship": "Diamond Princess",
}

# Territorios que primero venían como "país" y después como provincia de otro país.
TERRITORIO = {
    "Hong Kong": "China", "Hong Kong SAR": "China", "Macau": "China", "Macao SAR": "China",
    "Puerto Rico": "US", "Guam": "US",
    "Aruba": "Netherlands", "Curacao": "Netherlands",
    "Cayman Islands": "United Kingdom", "Channel Islands": "United Kingdom",
    "Gibraltar": "United Kingdom", "Jersey": "United Kingdom", "Guernsey": "United Kingdom",
    "Faroe Islands": "Denmark", "Greenland": "Denmark",
    "French Guiana": "France", "Guadeloupe": "France", "Martinique": "France", "Mayotte": "France",
    "Reunion": "France", "Saint Barthelemy": "France", "Saint Martin": "France", "St. Martin": "France",
}

for col in ["country_region", "province_state", "admin2"]:
    df[col] = df[col].str.strip()                       # quita espacios sobrantes (" Azerbaijan")

es_territorio = df["country_region"].isin(TERRITORIO)
df.loc[es_territorio, "province_state"] = df.loc[es_territorio, "country_region"]
df.loc[es_territorio, "country_region"] = df.loc[es_territorio, "country_region"].map(TERRITORIO)
df["country_region"] = df["country_region"].replace(PAIS)

### Cruce con la tabla UID
Cruzamos por **país + provincia + condado** y agregamos la columna `uid`: es la llave para traer
población, ISO3 y coordenadas en el DW. Provincia o condado vacío = `""` (el territorio es el país o la provincia completa).

In [ ]:
llave = ["country_region", "province_state", "admin2"]
tabla_uid = uid.rename(columns={"Country_Region": "country_region", "Province_State": "province_state",
                                "Admin2": "admin2", "UID": "uid"})[llave + ["uid"]]
tabla_uid = tabla_uid.fillna({"province_state": "", "admin2": ""})

filas_antes = len(df)
df = df.fillna({"province_state": "", "admin2": ""}).merge(tabla_uid, on=llave, how="left")
print("Filas antes:", filas_antes, "| después:", len(df))   # el cruce no debe duplicar filas

### Control: cuántas filas cruzan con la tabla UID

In [ ]:
cruza = df["uid"].notna()
print(f"Filas que cruzan: {cruza.mean():.2%} ({cruza.sum()} de {len(df)})")
print("Países que no existen en la tabla UID:", sorted(set(df["country_region"]) - set(uid["Country_Region"])) or "ninguno")

### Territorios que no cruzan
No se corrigen aquí: si se corrigen, marcan o excluyen se decide en R2 (H2.4).
La mayoría son ciudades de EE.UU. y Canadá de feb–mar 2020 (`Los Angeles, CA`), cruceros y repatriados.

In [ ]:
no_cruzan = (
    df.loc[~cruza].groupby(llave)
      .agg(filas=("fecha_reporte", "size"), desde=("fecha_reporte", "min"), hasta=("fecha_reporte", "max"))
      .sort_values("filas", ascending=False)
)
print("Territorios que no cruzan:", len(no_cruzan))
no_cruzan.head(30)

## 4 · Capa limpia en Parquet (H1.4)
Guardamos la tabla unificada en `data/clean/`. Los siguientes notebooks parten leyendo este archivo,
sin volver a leer los 1.143 reportes.

Se guarda con `compression="zstd"`: con la compresión por defecto de pandas (`snappy`) pesa ~99 MB, justo en el
límite de 100 MB por archivo de GitHub; con Zstandard baja a ~66 MB, se lee igual de rápido y pandas/pyarrow
lo leen sin instalar nada extra.

In [ ]:
import os

os.makedirs("../data/clean", exist_ok=True)
RUTA_PARQUET = "../data/clean/covid_limpio.parquet"

df.to_parquet(RUTA_PARQUET, index=False, compression="zstd")   # ~66 MB (sin zstd: ~99 MB)

print(f"Parquet guardado: {os.path.getsize(RUTA_PARQUET) / 2**20:.1f} MB")

### CSV frente a Parquet: tamaño en disco y tiempo de lectura
Guardamos la misma tabla como CSV temporal, medimos los dos archivos y después borramos el CSV
(no se sube al repo).

In [ ]:
import time

RUTA_CSV = "../data/clean/covid_limpio_temporal.csv"   # solo para comparar
df.to_csv(RUTA_CSV, index=False)

inicio = time.time()
pd.read_csv(RUTA_CSV)
tiempo_csv = time.time() - inicio

inicio = time.time()
pd.read_parquet(RUTA_PARQUET)
tiempo_parquet = time.time() - inicio

comparacion = pd.DataFrame({
    "formato": ["CSV", "Parquet"],
    "tamaño_MB": [os.path.getsize(RUTA_CSV) / 2**20, os.path.getsize(RUTA_PARQUET) / 2**20],
    "lectura_s": [tiempo_csv, tiempo_parquet],
}).round(1)

os.remove(RUTA_CSV)
comparacion

**Por qué Parquet:** guarda los datos por columna y comprimidos, por eso pesa mucho menos y se lee más rápido.
Además conserva los tipos: al leer el CSV las fechas vuelven como texto; al leer el Parquet vuelven como fecha.